This is a notebook for gaining a functional understanding of the pruning and quantization concepts by applying those to a simple neural network

This concepts will be further explored in future notebooks

Este notebook tiene como objetivo comprender de forma práctica los conceptos de pruning y cuantización aplicándolos a una red neuronal simple.

Estos conceptos se explorarán con mayor profundidad en cuadernos futuros.

In [1]:
# --- LIBRARIES ---
import torch                          # Main PyTorch library
import torch.nn as nn                 # For building neural networks
import torch.nn.utils.prune as prune  # For pruning weights
import torch.quantization             # For quantization
from sklearn.datasets import make_classification  # To create synthetic data
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import os                            # For measuring model file size

# --- 1. CREATE SYNTHETIC DATASET ---
# Create a binary classification dataset with 2000 samples and 20 features
X, y = make_classification(
    n_samples=2000, n_features=20, n_informative=10, random_state=42
)

# Standardize features for better model convergence
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Convert numpy arrays to PyTorch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.long)

# --- 2. DEFINE SIMPLE NEURAL NETWORK ---
class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(20, 64)  # First fully connected layer
        self.relu = nn.ReLU()         # Activation function (introduces non-linearity)
        self.fc2 = nn.Linear(64, 2)   # Output layer for 2 classes

    def forward(self, x):
        # Forward pass: input -> fc1 -> ReLU -> fc2
        return self.fc2(self.relu(self.fc1(x)))

# Instantiate the model
model = SimpleNet()

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()          # Suitable for classification tasks
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# --- 3. TRAIN THE BASE MODEL ---
for epoch in range(10):                   # Train for 10 epochs
    optimizer.zero_grad()                 # Reset gradients
    out = model(X_train)                  # Forward pass
    loss = criterion(out, y_train)        # Compute loss
    loss.backward()                       # Backpropagation
    optimizer.step()                      # Update weights

print("Base model trained ✅")

# --- 4. DEFINE EVALUATION FUNCTION ---
def evaluate(model):
    # Disable gradient computation during evaluation
    with torch.no_grad():
        preds = torch.argmax(model(X_test), dim=1)   # Predicted labels
        acc = (preds == y_test).float().mean().item() # Accuracy
    return acc

# Evaluate base model accuracy
base_acc = evaluate(model)
print(f"Base accuracy: {base_acc:.4f}")

# --- 5. APPLY PRUNING ---
# Define which layers and parameters to prune
parameters_to_prune = (
    (model.fc1, 'weight'),
    (model.fc2, 'weight'),
)

# Prune 40% of weights with the smallest absolute values (L1 criterion)
prune.global_unstructured(
    parameters_to_prune,
    pruning_method=prune.L1Unstructured,
    amount=0.4,
)

# Remove the pruning mask and make the weights permanent
for mod, name in parameters_to_prune:
    prune.remove(mod, name)

# Evaluate accuracy after pruning
pruned_acc = evaluate(model)
print(f"After pruning accuracy: {pruned_acc:.4f}")

# --- 6. APPLY QUANTIZATION (Dynamic) ---
# Switch model to evaluation mode before quantization
model.eval()

# Convert Linear layers to INT8 dynamically for faster inference
quantized_model = torch.quantization.quantize_dynamic(
    model, {nn.Linear}, dtype=torch.qint8
)

# Evaluate accuracy after quantization
quantized_acc = evaluate(quantized_model)
print(f"Quantized model accuracy: {quantized_acc:.4f}")

# --- 7. COMPARE MODEL SIZES ---
def model_size(model):
    # Save model temporarily and check its file size
    torch.save(model.state_dict(), "temp.pth")
    size = os.path.getsize("temp.pth") / 1e3  # Convert bytes to KB
    return size

print(f"Original model size: {model_size(model):.1f} KB")
print(f"Quantized model size: {model_size(quantized_model):.1f} KB")

# --- 8. SUMMARY ---
print("\n✅ Summary:")
print(f"Base acc: {base_acc:.4f}")
print(f"After pruning: {pruned_acc:.4f}")
print(f"After quantization: {quantized_acc:.4f}")


Base model trained ✅
Base accuracy: 0.8225
After pruning accuracy: 0.8275
Quantized model accuracy: 0.8225
Original model size: 8.1 KB
Quantized model size: 5.3 KB

✅ Summary:
Base acc: 0.8225
After pruning: 0.8275
After quantization: 0.8225


/tmp/ipython-input-519400873.py:100: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(
